

##  Arquitectura Orientada a Objetos en Cookiecutter

En lugar de scripts secuenciales o funciones sueltas, desacoplaremos la lógica en clases reutilizables siguiendo principios **SOLID**:

```text
src/
├── data/
│   └── make_dataset.py      <-- Clase `DataProcessor` (Carga, Limpieza, Guardado)
└── models/
    └── train_model.py       <-- Clase `ModelTrainer` e `Evaluator` (Entrenamiento y Métricas)
```

```text
  [ Tu Código OOP / Scripts / .dvc ] ----> Se guardan en -------> Git (GitHub / GitLab)
  [ Datasets / Imágenes / Modelos ] -----> Se guardan en -------> DVC (Google Drive Remote)
```

## Paso 1: Instalación de Dependencias e Inicialización de Git + DVC

In [ ]:
# Instalación de DVC con soporte oficial para Google Drive y librerías de ML
!pip install "dvc[gdrive]" pandas scikit-learn pyyaml pillow

In [ ]:
# Ajuste del directorio de trabajo a la raíz del proyecto
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    %cd ..
print(f"Directorio de trabajo actual: {os.getcwd()}")

In [ ]:
# 1. Inicializar Git y DVC
!git init
!dvc init

# 2. Guardar la configuración inicial de DVC en Git
!git add .dvc/ .gitignore
!git commit -m "build: initialize DVC and Git environment"

##  Paso 2: Configuración del Remote de Google Drive en DVC y Git

Copia el ID de la URL de tu carpeta de Google Drive:
`https://drive.google.com/drive/folders/`**`1A2b3C4d5E6f7G8h9I0j_kLmNoPqRsTuV`**

In [ ]:
GDRIVE_ID = "<TU_GDRIVE_FOLDER_ID>"  # <-- REEMPLAZAR CON TU ID REAL

# Registrar el remote de Google Drive
!dvc remote add -d gdrive_remote gdrive://{GDRIVE_ID}

# Registrar la configuración de DVC en Git
!git add .dvc/config
!git commit -m "config: set Google Drive as default DVC remote"

## Paso 3: Versionado de Datos (CSVs e Imágenes) con DVC y Git

Simularemos la generación de datos pesados y su registro tanto en DVC como en Git.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image

os.makedirs("data/raw/sample_images", exist_ok=True)

# Generar CSV sintético
df_raw = pd.DataFrame({
    'feature_1': np.random.normal(10, 2, 200),
    'feature_2': np.random.uniform(0, 100, 200),
    'is_fraud': np.random.choice([0, 1], size=200, p=[0.85, 0.15])
})
df_raw.to_csv("data/raw/transactions.csv", index=False)

# Generar Imágenes sintéticas
for i in range(5):
    img_array = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
    Image.fromarray(img_array).save(f"data/raw/sample_images/img_{i+1}.png")

print("✅ Datos pesados generados en data/raw/")

In [ ]:
# 1. Rastrear archivos pesados con DVC
!dvc add data/raw/transactions.csv
!dvc add data/raw/sample_images

# 2. Rastrear punteros pequeños (.dvc) y .gitignore con Git
!git add data/raw/transactions.csv.dvc data/raw/sample_images.dvc data/raw/.gitignore
!git commit -m "feat(data): track raw CSV and images directory pointers with DVC"

# 3. Subir datos a Google Drive
!dvc push

##  Paso 4: Implementación de Módulos Orientados a Objetos (OOP)

Escribiremos la lógica en clases Python dentro de la estructura `src/`.

In [ ]:
%%writefile params.yaml
prepare:
  split_ratio: 0.25
  random_state: 42

train:
  n_estimators: 150
  max_depth: 6
  target_col: "is_fraud"

In [ ]:
# Clase de Procesamiento de Datos (src/data/make_dataset.py)
os.makedirs("src/data", exist_ok=True)
with open("src/data/make_dataset.py", "w", encoding="utf-8") as f:
    f.write("""import pandas as pd
import os
import sys

class DataProcessor:
    """Clase responsable de la ingesta y preprocesamiento de datos."""
    def __init__(self, input_path: str, output_path: str):
        self.input_path = input_path
        self.output_path = output_path

    def load_data() -> pd.DataFrame:
        if not os.path.exists(self.input_path):
            raise FileNotFoundError(f"El archivo {self.input_path} no existe.")
        return pd.read_csv(self.input_path)

    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        # Transformaciones OOP limpias
        df_clean = df.dropna().copy()
        return df_clean

    def save_data(self, df: pd.DataFrame) -> None:
        os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
        df.to_csv(self.output_path, index=False)
        print(f"✅ [DataProcessor] Datos limpios guardados en: {self.output_path}")

    def run() -> None:
        df = self.load_data()
        df_clean = self.clean_data(df)
        self.save_data(df_clean)

if __name__ == '__main__':
    input_file = sys.argv[1] if len(sys.argv) > 1 else 'data/raw/transactions.csv'
    output_file = sys.argv[2] if len(sys.argv) > 2 else 'data/processed/clean_transactions.csv'
    
    processor = DataProcessor(input_file, output_file)
    processor.run()
""")

In [ ]:
# Clase de Entrenamiento y Evaluación (src/models/train_model.py)
os.makedirs("src/models", exist_ok=True)
with open("src/models/train_model.py", "w", encoding="utf-8") as f:
    f.write("""import pandas as pd
import yaml
import json
import os
import sys
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

class ModelTrainer:
    """Clase responsable del entrenamiento y evaluación del modelo."""
    def __init__(self, params_path: str, data_path: str, metrics_dir: str):
        self.params = self._load_params(params_path)
        self.data_path = data_path
        self.metrics_dir = metrics_dir
        self.model = None

    def _load_params(self, params_path: str) -> dict:
        with open(params_path, 'r') as f:
            return yaml.safe_load(f)

    def prepare_data():
        df = pd.read_csv(self.data_path)
        target_col = self.params['train']['target_col']
        X = df.drop(columns=[target_col])
        y = df[target_col]
        
        return train_test_split(
            X, y, 
            test_size=self.params['prepare']['split_ratio'], 
            random_state=self.params['prepare']['random_state']
        )

    def train(self, X_train, y_train) -> None:
        self.model = RandomForestClassifier(
            n_estimators=self.params['train']['n_estimators'],
            max_depth=self.params['train']['max_depth'],
            random_state=self.params['prepare']['random_state']
        )
        self.model.fit(X_train, y_train)
        print("✅ [ModelTrainer] Modelo entrenado exitosamente.")

    def evaluate(self, X_test, y_test) -> None:
        preds = self.model.predict(X_test)
        metrics = {
            'accuracy': float(accuracy_score(y_test, preds)),
            'precision': float(precision_score(y_test, preds, zero_division=0)),
            'recall': float(recall_score(y_test, preds, zero_division=0)),
            'f1_score': float(f1_score(y_test, preds, zero_division=0))
        }
        
        os.makedirs(self.metrics_dir, exist_ok=True)
        with open(os.path.join(self.metrics_dir, 'eval.json'), 'w') as f:
            json.dump(metrics, f, indent=4)
            
        plots_df = pd.DataFrame({'actual': y_test, 'predicted': preds})
        plots_df.to_csv(os.path.join(self.metrics_dir, 'plots.csv'), index=False)
        print(f"✅ [ModelTrainer] Métricas guardadas en: {self.metrics_dir}/")

    def run() -> None:
        X_train, X_test, y_train, y_test = self.prepare_data()
        self.train(X_train, y_train)
        self.evaluate(X_test, y_test)

if __name__ == '__main__':
    trainer = ModelTrainer('params.yaml', 'data/processed/clean_transactions.csv', 'metrics')
    trainer.run()
""")

## Paso 5: Declaración del Pipeline en `dvc.yaml`

In [ ]:
%%writefile dvc.yaml
stages:
  preprocess:
    cmd: python src/data/make_dataset.py data/raw/transactions.csv data/processed/clean_transactions.csv
    deps:
      - src/data/make_dataset.py
      - data/raw/transactions.csv
    outs:
      - data/processed/clean_transactions.csv

  train:
    cmd: python src/models/train_model.py
    deps:
      - src/models/train_model.py
      - data/processed/clean_transactions.csv
    params:
      - prepare.split_ratio
      - prepare.random_state
      - train.n_estimators
      - train.max_depth
      - train.target_col
    metrics:
      - metrics/eval.json:
          cache: false
    plots:
      - metrics/plots.csv:
          template: confusion
          x: predicted
          y: actual
          cache: false

## Paso 6: Ejecución, Control Sincronizado y Experimentos

In [ ]:
# Reproducir pipeline completo
!dvc repro

In [ ]:
# Guardar en Git los scripts OOP, pipeline, candados y métricas
!git add dvc.yaml dvc.lock params.yaml src/ metrics/ data/processed/.gitignore
!git commit -m "feat(pipeline): OOP pipeline execution for dataset processing and model training"

# Subir salidas pesadas a Google Drive
!dvc push

### Probar un Nuevo Experimento (Variación de Hiperparámetros)

In [ ]:
%%writefile params.yaml
prepare:
  split_ratio: 0.30
  random_state: 123

train:
  n_estimators: 300
  max_depth: 10
  target_col: "is_fraud"

In [ ]:
# Re-ejecutar solo las etapas afectadas
!dvc repro

In [ ]:
# Comparación de hiperparámetros y métricas con el estado guardado en Git
!dvc params diff
!dvc metrics diff HEAD

In [ ]:
# Registrar experimento en Git y DVC
!git add dvc.lock params.yaml metrics/
!git commit -m "experiment: increase estimators to 300 with max_depth 10"
!dvc push

## Paso 7: Flujo Colaborativo y Navegación entre Versiones

```bash
# Clonar código y punteros desde Git
git clone <URL_REPOSITORIO>
cd mi_proyecto

# Descargar datos desde Google Drive
dvc pull

# Cambiar de experimento/rama y sincronizar datos
git checkout HEAD~1
dvc checkout
```